In [1]:
import pandas as pd 
import numpy as np
import random
import math
import os
os.environ["OMP_NUM_THREADS"] = "1"
from sklearn.metrics.pairwise import haversine_distances


In [2]:
df = pd.read_excel('Input\\Sample data.xlsx')
print(f"Shape of df : {df.shape}")
print(f"df columns : {df.columns}")
df.head()

Shape of df : (72231, 29)
df columns : Index(['loan_id', 'client_id', 'loan_status_id', 'disbursedon_date',
       'closedon_date', 'expected_maturedon_date', 'address_region_value_id',
       'office_id_x', 'Branch', 'finflux_address_id', 'township',
       'ward_village', 'branch2', 'branch_code', 'village_latitude',
       'village_longitude', 'latitude_branch', 'longitude_branch',
       'repayment_schedule_id', 'due_date', 'assigned_loan_officer_id',
       'start_date', 'end_date', 'office_id_y', 'loan_officer',
       'employee_number', 'distance_km', 'bearing_deg', 'direction'],
      dtype='object')


,loan_id,client_id,loan_status_id,disbursedon_date,closedon_date,expected_maturedon_date,address_region_value_id,office_id_x,Branch,finflux_address_id,...,due_date,assigned_loan_officer_id,start_date,end_date,office_id_y,loan_officer,employee_number,distance_km,bearing_deg,direction
0,2177483,593231,300,2025-08-14,NaN,2026-08-10,449556,94,Yamethin,449556,...,2025-09-10,3630,NaN,NaN,94,"Lwin, Moe Aung -1_FO_RT",23091566,4.68,70.504558,E
1,2177130,493299,300,2025-08-13,NaN,2026-08-12,474618,114,Einme,474618,...,2025-09-10,4230,NaN,NaN,114,"Hnin, Ei Phyu - 3_FO_BT",24101345,28.15,204.145642,SW
2,2177095,139084,300,2025-08-14,NaN,2028-02-10,453266,64,Magway,453266,...,2025-09-10,4536,NaN,NaN,64,"Thae, Pwint Phue_FO_RT",25051516,24.27,76.540297,E
3,2177094,139014,300,2025-08-14,NaN,2027-08-10,453266,64,Magway,453266,...,2025-09-10,3977,NaN,NaN,64,"Ei, Shwe Sin -2_FO_RT",24031088,24.27,76.540297,E
4,2176750,593117,300,2025-08-13,NaN,2026-08-12,474602,114,Einme,474602,...,2025-09-10,4401,NaN,NaN,114,"Khant, Si Thu - 2_FO_BT",25011404,16.62,196.282498,S


In [3]:
print(f"df columns : {df.columns}")

df columns : Index(['loan_id', 'client_id', 'loan_status_id', 'disbursedon_date',
       'closedon_date', 'expected_maturedon_date', 'address_region_value_id',
       'office_id_x', 'Branch', 'finflux_address_id', 'township',
       'ward_village', 'branch2', 'branch_code', 'village_latitude',
       'village_longitude', 'latitude_branch', 'longitude_branch',
       'repayment_schedule_id', 'due_date', 'assigned_loan_officer_id',
       'start_date', 'end_date', 'office_id_y', 'loan_officer',
       'employee_number', 'distance_km', 'bearing_deg', 'direction'],
      dtype='object')


In [4]:
df = df[['client_id','finflux_address_id','village_latitude',
       'village_longitude', 'latitude_branch', 'longitude_branch', 
       'due_date', 'assigned_loan_officer_id','Branch']]
print(f"After selecting relevant columns : {df.columns}")
df.head()

After selecting relevant columns : Index(['client_id', 'finflux_address_id', 'village_latitude',
       'village_longitude', 'latitude_branch', 'longitude_branch', 'due_date',
       'assigned_loan_officer_id', 'Branch'],
      dtype='object')


,client_id,finflux_address_id,village_latitude,village_longitude,latitude_branch,longitude_branch,due_date,assigned_loan_officer_id,Branch
0,593231,449556,20.451191,96.183029,20.437149,96.140682,2025-09-10,3630,Yamethin
1,493299,474618,16.669941,95.070389,16.900944,95.178472,2025-09-10,4230,Einme
2,139084,453266,20.194349,95.168961,20.143691,94.942793,2025-09-10,4536,Magway
3,139014,453266,20.194349,95.168961,20.143691,94.942793,2025-09-10,3977,Magway
4,593117,474602,16.757484,95.134712,16.900944,95.178472,2025-09-10,4401,Einme


In [5]:
df = df[['client_id','village_latitude','village_longitude', 'due_date', 'Branch',
         'latitude_branch', 'longitude_branch','assigned_loan_officer_id' ]]
df.columns = ['client_id', 'client_latitude','client_longitude',  'due_date', 'Branch',
              'latitude_branch', 'longitude_branch','assigned_loan_officer_id']
df["due_date"] = pd.to_datetime(df["due_date"])

cycle_start = pd.Timestamp("2024-01-01") # this date can be adjusted as needed

df["cycle_day"] = (
    (df["due_date"] - cycle_start).dt.days % 28
) + 1

df.head()

,client_id,client_latitude,client_longitude,due_date,Branch,latitude_branch,longitude_branch,assigned_loan_officer_id,cycle_day
0,593231,20.451191,96.183029,2025-09-10,Yamethin,20.437149,96.140682,3630,3
1,493299,16.669941,95.070389,2025-09-10,Einme,16.900944,95.178472,4230,3
2,139084,20.194349,95.168961,2025-09-10,Magway,20.143691,94.942793,4536,3
3,139014,20.194349,95.168961,2025-09-10,Magway,20.143691,94.942793,3977,3
4,593117,16.757484,95.134712,2025-09-10,Einme,16.900944,95.178472,4401,3


In [6]:
Einme_df = df[df['Branch']=='Einme']
print(f"Shape of Einme_df : {Einme_df.shape}")

Shape of Einme_df : (5732, 9)


In [11]:
Einme_df['due_date'] = pd.to_datetime(Einme_df['due_date']).dt.date
Einme_df['due_date'] = Einme_df['due_date'].astype(str)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_30568\2527635481.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Einme_df['due_date'] = pd.to_datetime(Einme_df['due_date']).dt.date
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_30568\2527635481.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Einme_df['due_date'] = Einme_df['due_date'].astype(str)


In [13]:
Einme_df.groupby('due_date').agg({'assigned_loan_officer_id':'nunique'})

,assigned_loan_officer_id
due_date,
2025-09-02,7
2025-09-03,14
2025-09-04,14
2025-09-05,1
2025-09-09,10
2025-09-10,12
2025-09-11,14
2025-09-12,5
2025-09-15,8


In [8]:
office_location = Einme_df[['latitude_branch', 'longitude_branch']].drop_duplicates()
office_location['Name'] = 'Einme_Office'
office_location = office_location.rename(columns={'latitude_branch':'Latitude', 'longitude_branch':'Longitude'})
office_location['Type'] = 'Office'

In [ ]:
# make separate dataframes for each due_date from Einme_df
Einme_due_dates = Einme_df['due_date'].unique().tolist()
Einme_date_dfs = {}
for due_date in Einme_due_dates:
    Einme_date_dfs[due_date] = Einme_df[Einme_df['due_date']==due_date][['client_id', 'client_latitude', 'client_longitude']]
    Einme_date_dfs[due_date] = Einme_date_dfs[due_date].rename(columns={'client_id':'Name', 'client_latitude':'Latitude', 'client_longitude':'Longitude'})
    Einme_date_dfs[due_date]['Type'] = 'Client'
    Einme_date_dfs[due_date] = pd.concat([Einme_date_dfs[due_date], office_location[['Name', 'Latitude', 'Longitude', 'Type']]], ignore_index=True)
    # print(f"  Shape of Einme_date_dfs[{due_date}] : {Einme_date_dfs[due_date].shape}")
    Einme_date_dfs[due_date] = Einme_date_dfs[due_date].drop_duplicates(subset=['Latitude', 'Longitude'])
    print(f"Shape of Einme_date_dfs[{due_date}] : {Einme_date_dfs[due_date]}")
    print("____________________________________________________________")
    


KeyError: '2025-09-10'

In [ ]:
for due_date in Einme_due_dates:
    Einme_date_dfs[due_date].reset_index(drop=True, inplace=True)
    # export each date dataframe to csv 
    Einme_date_dfs[due_date].to_csv(f'Input\\Einme_clients_{due_date}.csv', index=False)
    print(f"Exported Einme_date_dfs[{due_date}] to Input\\Einme_clients_{due_date}.csv")

In [ ]:
Einme_date_dfs['2025-09-10'].head()